<a href="https://colab.research.google.com/github/catrina-llamas-1/Cats-Repository/blob/main/amenities_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amenities Matcher: Dataset A ↔ Dataset B (with Fuzzy Matching)

This notebook:
1. Loads Dataset A and Dataset B
2. Normalises amenity strings and **fuzzy-matches** them against a canonical list
3. **Fuzzy-matches** rows across datasets by address (handles abbreviation differences)
4. Writes matched canonical amenities back into Dataset A’s `amenities` column
5. Exports the updated Dataset A (retaining its original format)

**Fuzzy matching** handles:
- Minor spelling variants (`Bio-Tech/ Lab Space` → `Bio-Tech / Lab Space`)
- Case differences (`fitness center` → `Fitness Center`)
- Partial abbreviation differences in addresses (`Ave NW` → `Avenue Northwest`)
- Whitespace / punctuation inconsistencies

## 0. Configuration — Edit This Section

In [1]:
# ── File paths ───────────────────────────────────────────────────────────────────
DATASET_A_PATH = "Costar.xlsx"      # Supports .xlsx, .xls, or .csv
DATASET_B_PATH = "dataset_costar.xlsx"  # Supports .xlsx, .xls, or .csv
OUTPUT_PATH    = "dataset_a_updated.xlsx"  # Output will match Dataset A’s format

# ── Column names (adjust if your files use different headers) ────────────────────
A_ADDRESS_COL   = "Primary address"   # Address column in Dataset A
A_AMENITIES_COL = "Amenities"         # Amenities column in Dataset A

B_ADDRESS_COL   = "Property Address"  # Address column in Dataset B
B_AMENITIES_COL = "Amenities"         # Amenities column in Dataset B

# ── Delimiter config ────────────────────────────────────────────────────────────
# Delimiter separating multiple amenities in a single Dataset B cell.
B_AMENITIES_DELIMITER = ","
# Delimiter used when writing multiple amenities back to Dataset A.
OUTPUT_DELIMITER = ", "

# ── Overwrite behaviour ─────────────────────────────────────────────────────────
# True  → replace any existing value in A’s amenities column on a match
# False → only fill when A’s amenities cell is empty / NaN
OVERWRITE_EXISTING = True

# ── Fuzzy matching thresholds (0–100; higher = stricter) ─────────────────────
# Amenity fuzzy threshold: minimum similarity score for a raw amenity value
# to be accepted as a canonical match.
#   80 → catches minor typos and punctuation differences (recommended)
#   70 → broader; may introduce false positives
#   90 → very strict; only near-identical strings match
AMENITY_FUZZY_THRESHOLD = 80

# Address fuzzy threshold: minimum similarity score for two addresses to be
# treated as the same property.
#   90 → keeps false positives very low (recommended default)
#   85 → handles moderate abbreviation differences (Ave vs Avenue, NW vs Northwest)
#   80 → more permissive; verify fuzzy matches before trusting results
ADDRESS_FUZZY_THRESHOLD = 90

# Print every fuzzy-match decision to stdout (useful for threshold tuning).
VERBOSE_FUZZY = False

## 1. Canonical Amenities List

In [ ]:
CANONICAL_AMENITIES = {
    "24 Hour Access",
    "Atrium",
    "Banking",
    "Bicycle Storage",
    "Bio-Tech / Lab Space",
    "Car Charging Station",
    "Conference Center",
    "Controlled Access",
    "Courtyard",
    "Coworking Space",
    "Energy Star Labeled",
    "Fitness Center",
    "Food Service",
    "On Transit",
    "Private Terrace",
    "Property Manager on Site",
    "Restaurant",
    "Roof Terrace",
    "Security System",
    "Shower Facilities",
    "Signage",
    "Storage Space",
    "Tenant Lounge",
    "Waterfront",
}

# Explicit value conversions applied before fuzzy matching.
# Keys must be lowercase; values must be exact canonical strings.
# Add or remove entries here as your data evolves.
AMENITY_CONVERSIONS: dict[str, str] = {
    "food court":            "Food Service",
    "convenience store":     "Food Service",
    "conferencing facility": "Conference Center",
    "yard":                  "Courtyard",
    "monument signage":      "Signage",
}

print(f"Canonical list       : {len(CANONICAL_AMENITIES)} amenities.")
print(f"Explicit conversions : {len(AMENITY_CONVERSIONS)} rules.")

## 2. Install & Import Dependencies

In [3]:
%pip install rapidfuzz --quiet
print("rapidfuzz ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.4 MB/s eta 0:00:00
rapidfuzz ready.


## 3. Helper Functions

`normalise_address` is applied to **both** datasets identically before any
address comparison. It expands abbreviations and strips ordinal suffixes so
that formatting differences resolve to the same string:

| Raw input | After `normalise_address` |
|---|---|
| `8104 82nd Ave NW` | `8104 82 avenue northwest` |
| `8104 82 Avenue Northwest` | `8104 82 avenue northwest` |
| `8560 Roper Rd` | `8560 roper road` |
| `8560 ROPER ROAD` | `8560 roper road` |

An address match is a **hard prerequisite** — amenities are never transferred
unless a Dataset A address first matches a Dataset B address (exact or fuzzy).

In [ ]:
import os
import re

import pandas as pd
from rapidfuzz import fuzz, process


# ── File I/O ──────────────────────────────────────────────────────────────────────

def load_file(path: str) -> pd.DataFrame:
    """Load a CSV or Excel file into a DataFrame."""
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".csv":
        return pd.read_csv(path, dtype=str)
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path, dtype=str)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Use .csv, .xlsx, or .xls.")


# ── Address normalisation tables ──────────────────────────────────────────────────
# Applied to BOTH datasets so the same raw string always produces the same key.

# Street-type abbreviation → full form  (whole-token substitution)
_STREET_TYPES: dict[str, str] = {
    "ave":  "avenue",
    "av":   "avenue",
    "st":   "street",
    "rd":   "road",
    "blvd": "boulevard",
    "boul": "boulevard",
    "dr":   "drive",
    "ct":   "court",
    "ln":   "lane",
    "pl":   "place",
    "cr":   "crescent",
    "cres": "crescent",
    "wy":   "way",
    "hwy":  "highway",
    "pkwy": "parkway",
    "cir":  "circle",
}

# Directional abbreviation → full form  (whole-token substitution)
_DIRECTIONS: dict[str, str] = {
    "nw": "northwest",
    "ne": "northeast",
    "sw": "southwest",
    "se": "southeast",
    "n":  "north",
    "s":  "south",
    "e":  "east",
    "w":  "west",
}

# Matches ordinal suffixes: 1st, 2nd, 3rd, 4th … 22nd, 82nd, 103rd …
_ORDINAL_RE = re.compile(r'\b(\d+)(?:st|nd|rd|th)\b')


def normalise_address(addr) -> str:
    """
    Full address normalisation applied to BOTH datasets before any matching.

    Steps
    -----
    1. Lowercase, strip, remove punctuation (periods / commas in abbreviations)
    2. Remove ordinal suffixes  → '82nd' becomes '82', '1st' becomes '1'
    3. Expand street-type abbreviations token-by-token
       'Ave' → 'avenue', 'St' → 'street', 'Rd' → 'road', 'Blvd' → 'boulevard' …
    4. Expand directional abbreviations token-by-token
       'NW' → 'northwest', 'SW' → 'southwest', 'NE' → 'northeast' …
    5. Collapse whitespace to a single space

    Because the same function is applied to both datasets, formatting
    differences resolve to the same canonical string before comparison.
    Example:  '82nd Ave NW'  and  '82 Avenue Northwest'  both become
    '82 avenue northwest' and will score 100 on any similarity metric.
    """
    if pd.isna(addr):
        return ""

    text = str(addr).strip().lower()

    # 1. Remove punctuation common in abbreviations (e.g. 'Ave.' → 'Ave')
    text = text.replace(".", "").replace(",", "")

    # 2. Strip ordinal suffixes from numbers
    text = _ORDINAL_RE.sub(r"\1", text)

    # 3. Expand street-type abbreviations (whole-token, left-to-right)
    tokens = text.split()
    tokens = [_STREET_TYPES.get(t, t) for t in tokens]
    text   = " ".join(tokens)

    # 4. Expand directional abbreviations (whole-token)
    tokens = text.split()
    tokens = [_DIRECTIONS.get(t, t) for t in tokens]
    text   = " ".join(tokens)

    # 5. Collapse whitespace
    return re.sub(r"\s+", " ", text).strip()


def normalise_text(text: str) -> str:
    """Lowercase + collapse whitespace (used as fuzzy-match input for amenities)."""
    return re.sub(r"\s+", " ", str(text).strip().lower())


# ── Amenity parsing ───────────────────────────────────────────────────────────────

def parse_amenities(raw, delimiter: str) -> list[str]:
    """Split a raw amenities string into individual trimmed tokens."""
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [a.strip() for a in str(raw).split(delimiter) if a.strip()]


# ── Fuzzy matching ────────────────────────────────────────────────────────────────

def fuzzy_match_amenity(
    raw: str,
    canonical: set,
    threshold: int,
    conversions: dict | None = None,
    verbose: bool = False,
) -> str | None:
    """
    Map a raw amenity string to the best-matching canonical amenity.

    Resolution order
    ----------------
    1. Exact match (case-sensitive) — zero cost.
    2. Case-insensitive exact match against canonical.
    3. Explicit conversion lookup (case-insensitive key).
       e.g. 'Food Court' → 'Food Service', 'Yard' → 'Courtyard'.
       Conversions take priority over fuzzy matching so intentional
       remappings are never overridden by a higher-scoring fuzzy hit.
    4. Fuzzy token_sort_ratio against all canonical values.
       token_sort_ratio sorts words alphabetically before comparing, so
       word-order and punctuation differences score high.

    Returns the canonical string, or None if no match meets `threshold`.
    """
    # 1. Exact
    if raw in canonical:
        return raw

    # 2. Case-insensitive exact
    raw_lower = raw.lower()
    for c in canonical:
        if c.lower() == raw_lower:
            if verbose:
                print(f"  [amenity ci-exact] '{raw}' → '{c}'")
            return c

    # 3. Explicit conversion lookup
    if conversions:
        converted = conversions.get(raw_lower)
        if converted is not None:
            if verbose:
                print(f"  [amenity conversion] '{raw}' → '{converted}'")
            return converted

    # 4. Fuzzy
    canonical_list  = list(canonical)
    canonical_norms = [normalise_text(c) for c in canonical_list]

    result = process.extractOne(
        normalise_text(raw),
        canonical_norms,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=threshold,
    )
    if result is not None:
        _matched_norm, score, idx = result
        canon = canonical_list[idx]
        if verbose:
            print(f"  [amenity fuzzy {score:.0f}] '{raw}' → '{canon}'")
        return canon

    return None


def fuzzy_match_address(
    addr_norm: str,
    lookup: dict,
    threshold: int,
    verbose: bool = False,
) -> str | None:
    """
    Return the lookup key that best matches `addr_norm`.

    Both `addr_norm` and every key in `lookup` are already fully normalised
    by `normalise_address`, so differences like '82nd Ave NW' vs '82 Avenue
    Northwest' are resolved before this function is ever called.

    Resolution order
    ----------------
    1. Exact key lookup — O(1), catches all cases resolved by normalisation.
    2. rapidfuzz WRatio fuzzy match against all keys above `threshold`.
       WRatio combines multiple scoring strategies and handles residual
       differences that normalisation alone cannot eliminate.

    Returns the matching key string, or None if no match meets `threshold`.
    Amenities are ONLY transferred when this function returns a non-None value.
    """
    # 1. Exact (post-normalisation)
    if addr_norm in lookup:
        return addr_norm

    if not lookup:
        return None

    # 2. Fuzzy fallback
    keys = list(lookup.keys())
    result = process.extractOne(
        addr_norm,
        keys,
        scorer=fuzz.WRatio,
        score_cutoff=threshold,
    )
    if result is not None:
        matched_key, score, _ = result
        if verbose:
            print(f"  [address fuzzy {score:.0f}] '{addr_norm}' → '{matched_key}'")
        return matched_key

    return None


print("Helper functions loaded.")

In [ ]:
# Verify normalisation on representative examples before running the full pipeline.
examples = [
    ("8104 82nd Ave NW",         "8104 82 avenue northwest"),
    ("8104 82 Avenue Northwest",  "8104 82 avenue northwest"),
    ("8560 Roper Rd",             "8560 roper road"),
    ("4212 Gateway Blvd NW",     "4212 gateway boulevard northwest"),
    ("10335 95th St NW",         "10335 95 street northwest"),
    ("131 1st Ave",              "131 1 avenue"),
]

all_ok = True
print(f"{'Raw input':<35}  {'Normalised':<40}  {'OK?'}")
print("-" * 85)
for raw, expected in examples:
    result = normalise_address(raw)
    ok     = result == expected
    if not ok:
        all_ok = False
    print(f"{raw:<35}  {result:<40}  {'✓' if ok else '✗  expected: ' + expected}")

print()
print("✅ All normalisation checks passed." if all_ok else "⚠️  Some checks failed — review above.")

## 4. Load the Datasets

In [7]:
df_a = load_file(DATASET_A_PATH)
df_b = load_file(DATASET_B_PATH)

print(f"Dataset A: {df_a.shape[0]:,} rows × {df_a.shape[1]} columns")
print(f"Dataset B: {df_b.shape[0]:,} rows × {df_b.shape[1]} columns")
print()
print("Dataset A columns:", df_a.columns.tolist())
print("Dataset B columns:", df_b.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_pi.xlsx'

In [ ]:
print("── Dataset A (first 3 rows) ──")
display(df_a.head(3))

print("── Dataset B (first 3 rows) ──")
display(df_b.head(3))

## 5. Validate Required Columns

In [ ]:
def check_col(df, col, label):
    if col not in df.columns:
        raise KeyError(
            f"Column '{col}' not found in {label}.\n"
            f"Available columns: {df.columns.tolist()}\n"
            f"Update the Configuration section at the top of this notebook."
        )
    print(f"  \u2713 {label}: '{col}'")

print("Checking columns …")
check_col(df_a, A_ADDRESS_COL,   "Dataset A")
check_col(df_a, A_AMENITIES_COL, "Dataset A")
check_col(df_b, B_ADDRESS_COL,   "Dataset B")
check_col(df_b, B_AMENITIES_COL, "Dataset B")
print("All required columns present.")

## 6. Build Address → Amenities Lookup from Dataset B

**Pipeline for each Dataset B row:**
1. `normalise_address` converts the raw address to a canonical key
   (ordinals stripped, abbreviations expanded, lowercase)
2. Each amenity value is resolved in order: exact → case-insensitive exact →
   explicit conversion (`AMENITY_CONVERSIONS`) → fuzzy match
3. Only values that pass are stored under the normalised address key

Amenities from Dataset B are **never used** unless a Dataset A address
matches the corresponding key — the address match is a hard prerequisite.

In [ ]:
b_lookup: dict[str, list[str]] = {}
conversion_amenity_map: dict[str, str] = {}  # raw → canonical via explicit conversion
fuzzy_amenity_map: dict[str, str] = {}        # raw → canonical via fuzzy match

for _, row in df_b.iterrows():
    addr_key = normalise_address(row[B_ADDRESS_COL])
    if not addr_key:
        continue

    raw_amenities = parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)

    for raw in raw_amenities:
        canon = fuzzy_match_amenity(
            raw,
            CANONICAL_AMENITIES,
            AMENITY_FUZZY_THRESHOLD,
            conversions=AMENITY_CONVERSIONS,
            verbose=VERBOSE_FUZZY,
        )
        if canon is None:
            continue  # Below threshold — skip this amenity

        # Track how the mapping was resolved (for the summary in Section 7)
        if raw != canon:
            if raw.lower() in AMENITY_CONVERSIONS:
                conversion_amenity_map[raw] = canon
            else:
                fuzzy_amenity_map[raw] = canon

        if addr_key not in b_lookup:
            b_lookup[addr_key] = []
        if canon not in b_lookup[addr_key]:
            b_lookup[addr_key].append(canon)

addresses_with_amenities = sum(1 for v in b_lookup.values() if v)
print(f"Lookup built: {len(b_lookup):,} unique addresses from Dataset B.")
print(f"Addresses with ≥1 canonical amenity : {addresses_with_amenities:,}")
print(f"Explicit conversion mappings applied : {len(conversion_amenity_map):,}")
print(f"Fuzzy mappings applied               : {len(fuzzy_amenity_map):,}")

## 7. Amenity Mapping Summary

Shows how each non-exact raw value in Dataset B was resolved.
Review both tables — adjust `AMENITY_CONVERSIONS` or `AMENITY_FUZZY_THRESHOLD` if any mappings look wrong.

In [ ]:
if conversion_amenity_map:
    df_conv = pd.DataFrame(
        sorted(conversion_amenity_map.items()),
        columns=["raw_value_in_dataset_b", "mapped_to_canonical"],
    )
    print(f"── Explicit conversions ({len(df_conv)}) ──")
    display(df_conv)
else:
    print("No explicit conversion rules were triggered.")

print()

if fuzzy_amenity_map:
    df_fuzzy_amenities = pd.DataFrame(
        sorted(fuzzy_amenity_map.items()),
        columns=["raw_value_in_dataset_b", "mapped_to_canonical"],
    )
    print(f"── Fuzzy matches ({len(df_fuzzy_amenities)}) ──")
    display(df_fuzzy_amenities)
else:
    print("No fuzzy amenity mappings applied — all accepted values were exact matches.")

## 8. Update Dataset A's Amenities Column

**Address match is a hard prerequisite.** For each Dataset A row:

1. `normalise_address` is applied (same function and tables as Dataset B)
2. An exact key match is attempted first — O(1), no fuzzy cost
3. If no exact match, `fuzzy_match_address` (WRatio) searches all B keys
4. **If still no match → row is skipped; amenities are left unchanged**
5. Only on a confirmed match are the pre-built canonical amenities written back

In [ ]:
df_out = df_a.copy()

matched_count      = 0
unmatched_count    = 0
skipped_count      = 0
fuzzy_addr_matches = []  # (a_address, matched_b_key) pairs for the review below

for idx, row in df_out.iterrows():
    addr_norm = normalise_address(row[A_ADDRESS_COL])

    matched_key = fuzzy_match_address(
        addr_norm,
        b_lookup,
        ADDRESS_FUZZY_THRESHOLD,
        verbose=VERBOSE_FUZZY,
    )

    if matched_key is None:
        unmatched_count += 1
        continue

    # Record fuzzy (non-exact) address matches for transparency
    if matched_key != addr_norm:
        fuzzy_addr_matches.append({
            "dataset_a_address": row[A_ADDRESS_COL],
            "matched_dataset_b_key": matched_key,
        })

    canonical_amenities = b_lookup[matched_key]
    if not canonical_amenities:
        unmatched_count += 1
        continue

    existing  = row[A_AMENITIES_COL]
    has_value = not (pd.isna(existing) or str(existing).strip() == "")

    if has_value and not OVERWRITE_EXISTING:
        skipped_count += 1
        continue

    df_out.at[idx, A_AMENITIES_COL] = OUTPUT_DELIMITER.join(canonical_amenities)
    matched_count += 1

print("Results:")
print(f"  Rows updated                       : {matched_count:,}")
print(f"  No Dataset B match found           : {unmatched_count:,}")
print(f"  Skipped (OVERWRITE_EXISTING=False) : {skipped_count:,}")
print(f"  Fuzzy address matches used         : {len(fuzzy_addr_matches):,}")

## 9. Review Changes

In [ ]:
# Side-by-side: original vs updated amenities for every changed row
changed_mask = (
    df_out[A_AMENITIES_COL].fillna("") != df_a[A_AMENITIES_COL].fillna("")
)

comparison = pd.DataFrame({
    A_ADDRESS_COL       : df_a.loc[changed_mask, A_ADDRESS_COL],
    "amenities_original": df_a.loc[changed_mask, A_AMENITIES_COL],
    "amenities_updated" : df_out.loc[changed_mask, A_AMENITIES_COL],
})

print(f"{len(comparison):,} row(s) had their amenities updated.")
display(comparison.reset_index(drop=True))

In [ ]:
# Fuzzy address matches — inspect to confirm they are the same property
if fuzzy_addr_matches:
    print(f"{len(fuzzy_addr_matches)} fuzzy address match(es) used (verify these are correct):")
    display(pd.DataFrame(fuzzy_addr_matches))
else:
    print("All address matches were exact (no fuzzy address matching was needed).")

## 10. Export Updated Dataset A

In [ ]:
ext = os.path.splitext(OUTPUT_PATH)[-1].lower()

if ext == ".csv":
    df_out.to_csv(OUTPUT_PATH, index=False)
elif ext in (".xlsx", ".xls"):
    df_out.to_excel(OUTPUT_PATH, index=False)
else:
    raise ValueError(f"Unsupported output format: {ext}")

print(f"✅ Saved updated Dataset A to: {OUTPUT_PATH}")
print(f"   Shape: {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")

## 11. Diagnostic: Unmatched Dataset A Addresses

Addresses in Dataset A that could not be linked to any Dataset B row
(even after fuzzy matching). Consider lowering `ADDRESS_FUZZY_THRESHOLD`
or checking for systematic formatting differences.

In [ ]:
unmatched_mask = df_a[A_ADDRESS_COL].apply(
    lambda addr: fuzzy_match_address(
        normalise_address(addr), b_lookup, ADDRESS_FUZZY_THRESHOLD
    ) is None
)

unmatched_rows = df_a[unmatched_mask]
print(f"{len(unmatched_rows):,} Dataset A address(es) could not be matched to Dataset B:")
display(unmatched_rows[[A_ADDRESS_COL]].drop_duplicates().reset_index(drop=True))

## 12. Diagnostic: Amenity Values That Could Not Be Matched

Raw amenity strings from Dataset B that fell below `AMENITY_FUZZY_THRESHOLD`
and were therefore discarded. Review to decide whether to:
- **Add** the value to `CANONICAL_AMENITIES` (if it’s a valid amenity)
- **Lower** `AMENITY_FUZZY_THRESHOLD` (if the match is close but scores too low)
- **Leave** it out (if it truly has no canonical equivalent)

In [ ]:
from collections import Counter

still_unmatched: list[str] = []
for _, row in df_b.iterrows():
    for raw in parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER):
        if fuzzy_match_amenity(
            raw, CANONICAL_AMENITIES, AMENITY_FUZZY_THRESHOLD,
            conversions=AMENITY_CONVERSIONS,
        ) is None:
            still_unmatched.append(raw)

freq = Counter(still_unmatched)
df_still_unmatched = (
    pd.DataFrame(
        freq.items(),
        columns=["unmatched_value", "occurrences_in_dataset_b"],
    )
    .sort_values("occurrences_in_dataset_b", ascending=False)
    .reset_index(drop=True)
)

if df_still_unmatched.empty:
    print("✅ Every amenity value in Dataset B was matched to a canonical amenity.")
else:
    print(
        f"{len(df_still_unmatched):,} distinct value(s) in Dataset B could not be matched "
        f"(threshold={AMENITY_FUZZY_THRESHOLD}):"
    )
    display(df_still_unmatched)

## 13. (Optional) Interactive Threshold Tuning

Run this cell to see how the number of matched amenities and addresses
changes as you vary each threshold. Use it to find the sweet spot before
re-running the full pipeline with your chosen values.

In [ ]:
print("Amenity threshold sweep (unique raw values matched vs. total raw values):")
print(f"{'Threshold':>10}  {'Matched':>8}  {'Total':>8}  {'Match %':>8}")
print("-" * 44)

all_raw_amenities = [
    raw
    for _, row in df_b.iterrows()
    for raw in parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)
]
unique_raw = list(set(all_raw_amenities))
total = len(unique_raw)

for t in [60, 70, 75, 80, 85, 90, 95]:
    matched = sum(
        1 for r in unique_raw
        if fuzzy_match_amenity(
            r, CANONICAL_AMENITIES, t, conversions=AMENITY_CONVERSIONS
        ) is not None
    )
    pct = matched / total * 100 if total else 0
    marker = "  ← current" if t == AMENITY_FUZZY_THRESHOLD else ""
    print(f"{t:>10}  {matched:>8,}  {total:>8,}  {pct:>7.1f}%{marker}")

print()
print("Address threshold sweep (Dataset A rows matched to at least one Dataset B entry):")
print(f"{'Threshold':>10}  {'Matched':>8}  {'Total':>8}  {'Match %':>8}")
print("-" * 44)

a_norms = df_a[A_ADDRESS_COL].apply(normalise_address).tolist()
total_a = len(a_norms)

for t in [70, 75, 80, 85, 88, 90, 92, 95]:
    matched_a = sum(
        1 for addr in a_norms
        if fuzzy_match_address(addr, b_lookup, t) is not None
    )
    pct = matched_a / total_a * 100 if total_a else 0
    marker = "  ← current" if t == ADDRESS_FUZZY_THRESHOLD else ""
    print(f"{t:>10}  {matched_a:>8,}  {total_a:>8,}  {pct:>7.1f}%{marker}")